# 02_auth_override — Per-route `auth_coordinator` override

`BaseAdapter` requires one `auth_coordinator` — the strict default every route inherits. But some routes (a login endpoint, a public health check) legitimately need to bypass that default. Instead of weakening the adapter-wide coordinator for everyone, a single route can carry its own via `auth_coordinator=`.

**What's new**

| Concept | Description |
|---------|-------------|
| `BaseRouteRecord.auth_coordinator` | optional per-route override, `None` by default |
| `BaseAdapter.effective_auth_coordinator(record)` | returns the route's own coordinator when set, else the adapter default |
| `.post/.get/.put/.delete/.patch(path, Action, auth_coordinator=...)` | how `FastApiAdapter` exposes the override (same keyword on `McpAdapter.tool(...)`) |

This notebook uses FastAPI's in-process `TestClient` — no server, perfect for Colab.

> Install: `pip install aoa-fastapi-adapter`.

In [ ]:
from typing import Any

from fastapi.testclient import TestClient
from pydantic import Field

from aoa.action_machine.auth import GuestRole, NoAuthCoordinator
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseResult, ParamsStub
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.fastapi import FastApiAdapter

## A trivial guest action

`PingAction` takes no parameters (`ParamsStub`) and is open to anyone (`@check_roles(GuestRole)`). It knows nothing about HTTP or authentication — the override lives entirely on the adapter side.

In [ ]:
class SystemDomain(BaseDomain):
    name = "system"
    description = "System-level operations"


class PingResult(BaseResult):
    message: str = Field(description="Response message")


@meta(description="Health probe.", domain=SystemDomain)
@check_roles(GuestRole)
class PingAction(BaseAction[ParamsStub, PingResult]):

    @summary_aspect("Return a fixed pong message.")
    async def build_summary(self, params, state, box, connections) -> PingResult:
        _ = (params, state, box, connections)
        return PingResult(message="pong")

## A strict default, and one route that opts out

`DenyAllCoordinator` stands in for a real strict coordinator (JWT, API key, ...) that always rejects — it always returns `None`, which the adapter turns into `AuthorizationError` (HTTP 403). It becomes the adapter's *default*.

`/ping` inherits that default and gets rejected. `/ping-open` passes its own `auth_coordinator=NoAuthCoordinator(context=Context())` — a per-route override — and succeeds, without touching the default for any other route.

In [ ]:
class DenyAllCoordinator:
    """
    Stand-in for a real strict coordinator (JWT, API key, ...).

    Always returns ``None`` — the adapter turns that into ``AuthorizationError``
    (HTTP 403). Represents the adapter-wide default in this demo.
    """

    async def process(self, request_data: Any) -> Context | None:
        _ = request_data
        return None


def build_app():
    machine = ActionProductMachine()
    strict_default = DenyAllCoordinator()

    return (
        FastApiAdapter(machine=machine, auth_coordinator=strict_default, title="Auth Override Demo")
        .get("/ping", PingAction, tags=["system"])
        .get(
            "/ping-open",
            PingAction,
            tags=["system"],
            auth_coordinator=NoAuthCoordinator(context=Context()),
        )
        .build()
    )

## Run

`TestClient` drives the app in-process — no server, no `await`. `/ping` inherits the strict adapter default and is rejected (403); `/ping-open` uses its own override and succeeds (200).

In [ ]:
def main() -> None:
    client = TestClient(build_app())

    protected = client.get("/ping")
    print(f"GET /ping      (adapter default, strict) -> {protected.status_code} {protected.json()}")

    opened = client.get("/ping-open")
    print(f"GET /ping-open (route override, open)     -> {opened.status_code} {opened.json()}")


main()